In [ ]:
# =============================================================================
# CHAPTER 2: END-TO-END MACHINE LEARNING PROJECT — Setup & Data Split
# =============================================================================
#
# GOAL: Predict median house value for California districts using census data.
#
# WORKFLOW:
#   Cell 0 → Load data, stratified train/test split, separate features/labels
#   Cell 1 → Full preprocessing pipeline (24 engineered features)
#   Cell 2 → Train LinearRegression baseline, cross-validated RMSE
# =============================================================================

from pathlib import Path
import pandas as pd
import numpy as np
import tarfile
import urllib.request

# sklearn — data splitting & evaluation
from sklearn.model_selection import train_test_split, cross_val_score

# sklearn — imputation & encoding
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer

# sklearn — pipeline & composition
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer, make_column_selector

# sklearn — clustering & similarity (used in ClusterSimilarity)
from sklearn.cluster import KMeans
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics.pairwise import rbf_kernel

# sklearn — models
from sklearn.linear_model import LinearRegression


# --- Load data ---
# Dataset: California Housing (20,640 districts, 1990 census)
# Downloads and extracts from GitHub on first run, then reads from local CSV.

def load_housing_data():
    tarball_path = Path("datasets/housing.tgz")
    if not tarball_path.is_file():
        Path("datasets").mkdir(parents=True, exist_ok=True)
        url = "https://github.com/ageron/data/raw/main/housing.tgz"
        urllib.request.urlretrieve(url, tarball_path)
        with tarfile.open(tarball_path) as housing_tarball:
            housing_tarball.extractall(path="datasets")
    return pd.read_csv(Path("datasets/housing/housing.csv"))


housing_full = load_housing_data()


# --- Stratified train/test split ---
# median_income is the strongest predictor (+0.69 correlation with target).
# A random split risks skewing its distribution by chance.
# Binning into 5 income categories and using stratify= ensures both sets
# are proportionally representative of the full income range.

housing_full["income_cat"] = pd.cut(
    housing_full["median_income"],
    bins=[0., 1.5, 3.0, 4.5, 6., np.inf],
    labels=[1, 2, 3, 4, 5]
)

strat_train_set, strat_test_set = train_test_split(
    housing_full, test_size=0.2, stratify=housing_full["income_cat"], random_state=42
)
for set_ in (strat_train_set, strat_test_set):
    set_.drop("income_cat", axis=1, inplace=True)

# Save test set to disk and remove from memory — not touched until final eval.
strat_test_set.to_csv("datasets/housing/test_set.csv", index=False)
del strat_test_set


# --- Separate features and target ---
# housing        → feature matrix (9 columns, includes 1 categorical)
# housing_labels → target vector  (median_house_value)

housing = strat_train_set.drop("median_house_value", axis=1)
housing_labels = strat_train_set["median_house_value"].copy()

missing = housing.isnull().sum()
missing = missing[missing > 0]
print(f"Training rows  : {housing.shape[0]:,}")
print(f"Features       : {housing.shape[1]}")
print(f"Missing values : {', '.join(f'{col} ({n})' for col, n in missing.items())}")

In [ ]:
# =============================================================================
# FULL PREPROCESSING PIPELINE
# =============================================================================
#
# Consolidates all transformations arrived at through experimentation.
#
# FEATURE ENGINEERING DECISIONS:
#   - Ratio features (bedrooms, rooms_per_house, people_per_house):
#       Raw counts correlate poorly; ratios reveal density → better predictor.
#   - Log transform (bedrooms, rooms, population, households, income):
#       Long right-tails compressed to roughly Gaussian.
#       housing_median_age already roughly uniform — no log needed.
#   - Cluster similarity (latitude, longitude):
#       10 KMeans centroids capture "Bay Area effect", "LA effect" etc.
#       as smooth continuous features instead of raw coordinates.
#   - One-hot (ocean_proximity):
#       Nominal category, no natural ordering → binary columns.
#   - Remainder (housing_median_age):
#       Imputed + scaled, no other engineering needed.
#
# OUTPUT: (16512, 24)
#   3 ratio + 5 log + 10 cluster + 5 one-hot + 1 remainder
# =============================================================================


# --- ClusterSimilarity (custom sklearn transformer) ---
# Learns k geographic cluster centroids via KMeans, then computes RBF
# (Gaussian) similarity of each district to each centroid.
#   fit()       → KMeans on lat/lon → learns k centroids
#   transform() → rbf_kernel(X, centroids) → similarity matrix
#   K(x, c) = exp(-gamma * ||x - c||²)  →  1.0 at centroid, ~0.0 far away
# Why not raw lat/lon? A linear model can't interpret coordinates as spatial
# proximity. Cluster similarity gives it 10 smooth "nearness" features.

class ClusterSimilarity(BaseEstimator, TransformerMixin):
    def __init__(self, n_clusters=10, gamma=1.0, random_state=None):
        self.n_clusters = n_clusters
        self.gamma = gamma
        self.random_state = random_state

    def fit(self, X, y=None, sample_weight=None):
        self.kmeans_ = KMeans(self.n_clusters, random_state=self.random_state)
        self.kmeans_.fit(X, sample_weight=sample_weight)
        return self

    def transform(self, X):
        return rbf_kernel(X, self.kmeans_.cluster_centers_, gamma=self.gamma)

    def get_feature_names_out(self, names=None):
        return [f"Cluster {i} similarity" for i in range(self.n_clusters)]


# --- Ratio pipeline helper ---
# FunctionTransformer(column_ratio) wraps a plain function as a sklearn
# transformer so it can live inside a Pipeline/ColumnTransformer.
#   feature_names_out=ratio_name → custom naming function for the output column
# Each ratio pipeline: impute NaN → compute X[:,0]/X[:,1] → standardize

def column_ratio(X):
    return X[:, [0]] / X[:, [1]]

def ratio_name(function_transformer, feature_names_in):
    return ["ratio"]

def ratio_pipeline():
    return make_pipeline(
        SimpleImputer(strategy="median"),
        FunctionTransformer(column_ratio, feature_names_out=ratio_name),
        StandardScaler(),
    )


# --- Log pipeline ---
# FunctionTransformer(np.log) applies log element-wise.
#   feature_names_out="one-to-one" → output names match input names
# Impute first (log of NaN would propagate), then log, then standardize.
# Compresses long right-tails → roughly Gaussian → better for linear models.

log_pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    FunctionTransformer(np.log, feature_names_out="one-to-one"),
    StandardScaler(),
)


# --- Categorical pipeline ---
# SimpleImputer(strategy="most_frequent") fills any NaN with the mode.
# OneHotEncoder creates one binary column per category (5 for ocean_proximity).
#   Chosen over OrdinalEncoder: ocean_proximity has NO natural ordering —
#   integers would imply false magnitude (ISLAND=2 is not "twice" INLAND=1).
#   handle_unknown="ignore" → unseen categories at inference → all zeros
#   rather than raising an error.

cat_pipeline = make_pipeline(
    SimpleImputer(strategy="most_frequent"),
    OneHotEncoder(handle_unknown="ignore"),
)


# --- Default numeric pipeline ---
# SimpleImputer(strategy="median") fills NaN with training median.
#   Why median over mean? Robust to outliers (a few $500k-capped values
#   would skew the mean but not the median).
#   Why not drop NaN rows? At inference you can't drop — must handle consistently.
# StandardScaler standardizes to mean=0, std=1 (z-score).
#   Chosen over MinMaxScaler: less sensitive to outliers (outliers shift
#   min/max for ALL rows in MinMaxScaler). Most features roughly Gaussian.
#   Always fit on training data only — apply same params to new data.

default_num_pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
)


# --- Cluster similarity instance ---
# 10 clusters, gamma=1.0 (controls RBF bell-curve width)

cluster_simil = ClusterSimilarity(n_clusters=10, gamma=1., random_state=42)


# --- ColumnTransformer: assemble everything ---
# Applies different pipelines to different column subsets in parallel,
# then concatenates all outputs into a single NumPy array.
#   remainder=default_num_pipeline → catches housing_median_age (the only
#   column not explicitly assigned) and imputes + scales it, rather than
#   silently dropping it.
# make_column_selector(dtype_include=object) dynamically selects all
#   object-dtype columns (i.e. categorical). More robust than hardcoding
#   column names if the schema ever changes.

preprocessing = ColumnTransformer([
    ("bedrooms",         ratio_pipeline(), ["total_bedrooms", "total_rooms"]),
    ("rooms_per_house",  ratio_pipeline(), ["total_rooms", "households"]),
    ("people_per_house", ratio_pipeline(), ["population", "households"]),
    ("log",              log_pipeline,     ["total_bedrooms", "total_rooms",
                                            "population", "households", "median_income"]),
    ("geo",              cluster_simil,    ["latitude", "longitude"]),
    ("cat",              cat_pipeline,     make_column_selector(dtype_include=object)),
], remainder=default_num_pipeline)

housing_prepared = preprocessing.fit_transform(housing)

print(f"Output shape : {housing_prepared.shape}")
print(f"\nFeature names:")
for name in preprocessing.get_feature_names_out():
    print(f"  {name}")

In [ ]:
# =============================================================================
# TRAIN & EVALUATE
# =============================================================================
#
# Establishes a performance baseline before trying complex models.
# Cross-validated RMSE is the honest metric — training RMSE alone is misleading
# because the model has already seen the training data.
# =============================================================================


# --- Linear Regression baseline ---
# sklearn.linear_model.LinearRegression
#   Fits a hyperplane by minimizing residual sum of squares (OLS).
#   No hyperparameters to tune — the simplest possible baseline.
#   If a complex model can't beat this, something is wrong upstream.
#
# make_pipeline(preprocessing, LinearRegression())
#   Wraps preprocessing + model together. During cross-validation,
#   preprocessing.fit() runs ONLY on training folds — no data leakage.

lin_reg = make_pipeline(preprocessing, LinearRegression())
lin_reg.fit(housing, housing_labels)


# --- Cross-validated RMSE (10-fold) ---
# cross_val_score splits the training set into 10 folds:
#   Trains on 9, evaluates on 1, rotates through all 10.
#   Averages the 10 scores → stable estimate of generalization error.
#
# scoring="neg_root_mean_squared_error":
#   sklearn convention: higher = better, so RMSE is negated. We negate back.
#
# RMSE is in the same units as the target (dollars).
#   Penalizes large errors more than MAE (squares before rooting).
#   This baseline is the floor to beat with RandomForest, SVR, etc.

rmse_scores = -cross_val_score(
    lin_reg, housing, housing_labels,
    scoring="neg_root_mean_squared_error", cv=10
)

print(f"CV RMSE scores : {rmse_scores.round(0)}")
print(f"Mean RMSE      : ${rmse_scores.mean():,.0f}")
print(f"Std RMSE       : ${rmse_scores.std():,.0f}")


# --- Sample predictions vs actuals ---

sample = housing.iloc[:5]
predictions = lin_reg.predict(sample)

print("\nSample predictions vs actuals:")
pd.DataFrame({
    "actual":    housing_labels.iloc[:5].values,
    "predicted": predictions.round(0),
    "error":     (predictions - housing_labels.iloc[:5].values).round(0),
})